# Outlook 连接测试
测试通过 `win32com.client` 连接本地 Outlook 客户端。

In [1]:
# Cell 1: 检查 pywin32 是否已安装
try:
    import win32com.client
    print("✓ pywin32 已安装")
except ImportError:
    print("✗ pywin32 未安装，请运行: pip install pywin32")

✓ pywin32 已安装


In [2]:
# Cell 2: 连接 Outlook 并获取账户信息
import win32com.client

try:
    outlook = win32com.client.Dispatch("Outlook.Application")
    namespace = outlook.GetNamespace("MAPI")
    
    print("✓ 成功连接到 Outlook!")
    print(f"\n已配置的账户：")
    for account in namespace.Accounts:
        print(f"  - {account.DisplayName} ({account.SmtpAddress})")
except Exception as e:
    print(f"✗ 连接失败: {e}")

✓ 成功连接到 Outlook!

已配置的账户：
  - zengsh@westfund.com.au (zengsh@westfund.com.au)


In [3]:
# Cell 3: 读取收件箱最近 5 封邮件
try:
    inbox = namespace.GetDefaultFolder(6)  # 6 = olFolderInbox
    messages = inbox.Items
    messages.Sort("[ReceivedTime]", True)  # 按时间倒序
    
    print(f"收件箱共有 {inbox.Items.Count} 封邮件")
    print("\n最近 5 封邮件：")
    for i, msg in enumerate(messages):
        if i >= 5:
            break
        print(f"  [{i+1}] {msg.ReceivedTime:%Y-%m-%d %H:%M}  |  {msg.SenderName:<30}  |  {msg.Subject}")
except Exception as e:
    print(f"✗ 读取收件箱失败: {e}")

收件箱共有 860 封邮件

最近 5 封邮件：
  [1] 2026-03-18 15:13  |  Shawn Zeng                      |  Outlook 连接测试
  [2] 2026-03-18 11:18  |  Gustavo Lummertz                |  Data Governance Committee
  [3] 2026-03-18 11:18  |  Gustavo Lummertz                |  Data Governance Committee
  [4] 2026-03-18 11:09  |  Gustavo Lummertz                |  Re: Portfolio Dashboard – Deployment Approval Request
  [5] 2026-03-18 10:32  |  Magdalena Herceg                |  AAGG Meeting #6


In [10]:
# Cell 4: 阅读指定邮件的详细内容
# 修改 READ_INDEX 来选择第几封邮件（0 = 最新一封）
READ_INDEX = 7

try:
    inbox = namespace.GetDefaultFolder(6)
    messages = inbox.Items
    messages.Sort("[ReceivedTime]", True)

    msg = messages[READ_INDEX]

    print("=" * 60)
    print(f"主题:    {msg.Subject}")
    print(f"发件人:  {msg.SenderName} <{msg.SenderEmailAddress}>")
    print(f"收件人:  {msg.To}")
    print(f"抄送:    {msg.CC or '（无）'}")
    print(f"时间:    {msg.ReceivedTime:%Y-%m-%d %H:%M:%S}")
    print(f"附件数:  {msg.Attachments.Count}")
    if msg.Attachments.Count > 0:
        for att in msg.Attachments:
            print(f"  - {att.FileName}")
    print("=" * 60)
    print("\n【正文内容】\n")

    # 优先显示纯文本正文；如果为空则从 HTML 中提取
    body = msg.Body.strip()
    if body:
        print(body[:3000])  # 最多显示 3000 字符，避免超长邮件刷屏
        if len(msg.Body.strip()) > 3000:
            print(f"\n... （正文已截断，共 {len(body)} 字符）")
    else:
        # 从 HTML Body 简单提取文字
        import re
        html_body = msg.HTMLBody or ""
        plain = re.sub(r"<[^>]+>", "", html_body)
        plain = re.sub(r"\s+", " ", plain).strip()
        print(plain[:3000])

except Exception as e:
    print(f"✗ 读取邮件失败: {e}")

主题:    RE: OVHC Go-Live Celebration
发件人:  Lynton Till </O=EXCHANGELABS/OU=EXCHANGE ADMINISTRATIVE GROUP (FYDIBOHF23SPDLT)/CN=RECIPIENTS/CN=6ED6C5631C5242FAA312C7CCF0C5488E-55C99416-EE>
收件人:  DL Clarence Street
抄送:    Jessica Harrowsmith; Gustavo Lummertz; Richard Ferchow; Anna Maltabarow; Tarnee Jacobson; Magdalena Herceg; Ilia Vavilin; Carolina Gutierrez; Alexandra Blakely; Liz Casmiri; Brendan Mercieca; Stephen Smart; John Iverach; Mario Fortunato; Rachel Rees; Raihan Siddiquee; Mark Genovese; Zach Layton; Ryanz Prasad; Trevan Spiteri; Vedanti Ladda; Daniel McLoughlan; Fiona Goldrick; Minta Anto
时间:    2026-03-17 11:51:26
附件数:  1
  - image001.jpg

【正文内容】

Hi Everyone,

 

A reminder that the OVHC project is providing Lunch in the Clarence St Office Kitchen to celebrate the launch of the new OVHC products with everyone in the business and thank those involved. Lunch is served – please enjoy!

 

Based on acceptances I’ve catered for 15 people – but if you happen to be there today you’

In [ ]:
# Cell 4: （可选）发送一封测试邮件 — 取消注释后运行
# 请先修改收件人地址再运行！

# TO_ADDRESS = "your.email@example.com"  # <-- 改成你自己的地址

# mail = outlook.CreateItem(0)  # 0 = olMailItem
# mail.To = TO_ADDRESS
# mail.Subject = "Outlook 连接测试"
# mail.Body = "这是一封来自 Python 的测试邮件。"
# mail.Send()
# print(f"✓ 测试邮件已发送至 {TO_ADDRESS}")